In [18]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

# Load dataset
df = sns.load_dataset('tips')

# === Feature Engineering ===

# Tip percentage
df['tip_pct'] = round(df['tip'] / df['total_bill'], 2)

# Tip level
bins = [0, 0.1, 0.2, df['tip_pct'].max()]
labels = ['Low', 'Medium', 'High']
df['tip_level'] = pd.cut(df['tip_pct'], bins=bins, labels=labels)

# Interaction feature
df['bill_size_interaction'] = df['total_bill'] * df['size']

# Convert categorical columns to string before combining
df['sex'] = df['sex'].astype(str)
df['time'] = df['time'].astype(str)
df['sex_time'] = df['sex'] + "_" + df['time']

# One-hot encoding
df = pd.get_dummies(df, columns=['sex_time'], drop_first=True)

# Drop unused or irrelevant columns
df.drop(columns=['smoker', 'day'], inplace=True)

# === Prepare Data for Model ===

features = ['total_bill', 'tip', 'size', 'bill_size_interaction'] + [col for col in df.columns if col.startswith('sex_time_')]
X = df[features]
y = df['tip_level']

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Save feature names for reuse
model_features = X.columns.tolist()

# Train
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Save model and encoder
joblib.dump(model, 'logreg_model.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(model_features, 'model_features.pkl')

# Load model & tools
import joblib

model = joblib.load('logreg_model.pkl')
le = joblib.load('label_encoder.pkl')
model_features = joblib.load('model_features.pkl')

# New sample data
new_data = pd.DataFrame({
    'total_bill': [30.5],
    'tip': [5.0],
    'size': [2],
})

# Feature engineering for new data
new_data['tip_pct'] = round(new_data['tip'] / new_data['total_bill'], 2)
new_data['bill_size_interaction'] = new_data['total_bill'] * new_data['size']
new_data['sex_time'] = 'Male_Dinner'  # Example input

# One-hot encoding sex_time
sex_time_dummies = pd.get_dummies(new_data['sex_time'], prefix='sex_time')
new_data = pd.concat([new_data, sex_time_dummies], axis=1)

# Align columns with training data
for col in model_features:
    if col not in new_data.columns:
        new_data[col] = 0  # Fill missing dummy columns

# Ensure correct column orders
X_new = new_data[model_features]

# Predict
pred = model.predict(X_new)
print("Predicted Tip Level:", le.inverse_transform(pred)[0])


Predicted Tip Level: Medium
